# Week 4: Transfer Learning, BERT (Homework)

## Question Search Engine

Embeddings are a good source of information for solving various tasks. For example, we can classify texts or find similar documents using their representations. We already know about word2vec, GloVe and fasttext, but they don't use context information from given text (only from contexts of source data).

For today we will use full power of context-aware embeddings to find text duplicates!

__Warning:__ this task assumes you have seen `seminar.ipynb`!

In [33]:
%pip install --upgrade transformers datasets accelerate deepspeed
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
import datasets

### Data Preparation

In [34]:
qqp = datasets.load_dataset("SetFit/qqp")
print("\n")
print("Sample[0]:", qqp["train"][0])
print("Sample[3]:", qqp["train"][3])

Repo card metadata block was not found. Setting CardData to empty.




Sample[0]: {'text1': 'How is the life of a math student? Could you describe your own experiences?', 'text2': 'Which level of prepration is enough for the exam jlpt5?', 'label': 0, 'idx': 0, 'label_text': 'not duplicate'}
Sample[3]: {'text1': 'What can one do after MBBS?', 'text2': 'What do i do after my MBBS ?', 'label': 1, 'idx': 3, 'label_text': 'duplicate'}


In [35]:
model_name = "gchhablani/bert-base-cased-finetuned-qqp"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)

In [36]:
MAX_LENGTH = 128

def preprocess_function(examples):
    result = tokenizer(
        examples["text1"],
        examples["text2"],
        padding="max_length",
        max_length=MAX_LENGTH,
        truncation=True,
    )

    result["label"] = examples["label"]

    return result

In [37]:
qqp_preprocessed = qqp.map(preprocess_function, batched=True)

Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

In [38]:
print(repr(qqp_preprocessed["train"][0]["input_ids"])[:100], "...")

[101, 1731, 1110, 1103, 1297, 1104, 170, 12523, 2377, 136, 7426, 1128, 5594, 1240, 1319, 5758, 136,  ...


### Evaluation (1 point)

We randomly chose a model trained on QQP - but is it any good?

One way to measure this is with validation accuracy - which is what you will implement next.

Here's the interface to help you do that:

In [39]:
val_set = qqp_preprocessed["validation"]
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=1, shuffle=False, collate_fn=transformers.default_data_collator
)

In [40]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device).eval()

for batch in val_loader:
    batch = {k: v.to(device) for k, v in batch.items()}
    break  # here be your training code
print("Sample batch:", batch)

with torch.no_grad():
    predicted = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        token_type_ids=batch["token_type_ids"],
    )

print("\nPrediction (probs):", torch.softmax(predicted.logits, dim=1).cpu().numpy())

Sample batch: {'labels': tensor([0], device='cuda:0'), 'idx': tensor([0], device='cuda:0'), 'input_ids': tensor([[  101,  2009,  1132,  2170,   118,  4038,  1177,  2712,   136,   102,
          2009,  1132,  1117, 10224,  4724,  1177,  2712,   136,   102,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,    

**Task 1 (1 point)**

- Measure the validation accuracy of your model. Doing so naively may take several hours. Please make sure you use the following optimizations:
  - Run the model on GPU with no_grad
  - Using batch size larger than 1
  - Use optimize data loader with num_workers > 1
  - (Optional) Use [mixed precision](https://pytorch.org/docs/stable/notes/amp_examples.html)


In [42]:
def evaluate(model, val_loader):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device).eval()
    torch.set_grad_enabled(False)

    correct = 0
    total = 0

    for batch in val_loader:
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}

        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            token_type_ids=batch["token_type_ids"]
        )

        preds = outputs.logits.argmax(dim=1)
        labels = batch["labels"]
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    torch.set_grad_enabled(True)

    accuracy = correct / total
    print(accuracy)
    return accuracy

accuracy = evaluate(model, val_loader)

0.9083848627256987


In [43]:
assert 0.9 < accuracy < 0.91

### Training (4 points)

For this task, you have two options:

__Option A:__ fine-tune your own model. You are free to choose any model __except for the original BERT.__ We recommend [DeBERTa-v3](https://huggingface.co/microsoft/deberta-v3-base). Better yet, choose the best model based on public benchmarks (e.g. [GLUE](https://gluebenchmark.com/)).

You can write the training code manually or use transformers.Trainer (see [this example](https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification)). Please make sure that your model's accuracy is at least __comparable__ with the above example for BERT.


__Option B:__ compare at least 3 pre-finetuned models (in addition to the above BERT model). For each model, report (1) its accuracy, (2) its speed, measured in samples per second in your hardware setup and (3) its size in megabytes. Please take care to compare models in equal setting, e.g. same CPU / GPU. Compile your results into a table and write a short (~half-page on top of a table) report, summarizing your findings.

**Task 2 (4 points)**
- Choose Option A or Option B (only one will be graded)
- Follow all the instructions and restrictions

In [45]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

bert_base_name = "textattack/bert-base-uncased-QQP"
tokenizer2 = AutoTokenizer.from_pretrained(bert_base_name)
model2 = AutoModelForSequenceClassification.from_pretrained(bert_base_name)

roberta_name = "JeremiahZ/roberta-base-qqp"
tokenizer3 = AutoTokenizer.from_pretrained(roberta_name)
model3 = AutoModelForSequenceClassification.from_pretrained(roberta_name)

distillbert_name = "textattack/distilbert-base-uncased-QQP"
tokenizer4 = AutoTokenizer.from_pretrained(distillbert_name)
model4 = AutoModelForSequenceClassification.from_pretrained(distillbert_name)

In [49]:

import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import time

@torch.no_grad()
def evaluate_best(model, tokenizer, q1, q2, y_true, batch_size=64, max_length=MAX_LENGTH):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device).eval()
    model_size = sum(p.numel() for p in model.parameters()) * 4 / (1024 * 1024)

    preds = []
    start_time = time.time()

    for i in range(0, len(q1), batch_size):
        enc = tokenizer(
            q1[i:i+batch_size],
            q2[i:i+batch_size],
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        enc = {k: v.to(device) for k, v in enc.items()}

        logits = model(**enc).logits
        preds.extend(logits.argmax(dim=-1).cpu().numpy().tolist())

    end_time = time.time()
    total_samples = len(q1)
    inference_time = end_time - start_time
    samples_per_second = total_samples / inference_time

    acc = accuracy_score(y_true, preds)
    prec = precision_score(y_true, preds, zero_division=0)
    rec = recall_score(y_true, preds, zero_division=0)
    f1 = f1_score(y_true, preds, zero_division=0)

    return {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "samples_per_second": samples_per_second,
        "model_size_mb": model_size,
        "inference_time_seconds": inference_time
    }


In [50]:
import pandas as pd

q1, q2, y = qqp["validation"]["text1"], qqp["validation"]["text2"], qqp["validation"]["label"]

models_to_compare = [
    ("bert-base-cased-finetuned-qqp", tokenizer, model),
    ("bert-base-uncased QQP", tokenizer2, model2),
    ("roberta-base QQP",      tokenizer3, model3),
    ("distilbert-base QQP",   tokenizer4, model4),
]

rows = []
for name, tok, mdl in models_to_compare:
    m = evaluate_best(mdl, tok, q1, q2, y)
    rows.append({"model": name, **{k: round(v, 4) for k, v in m.items()}})

pd.DataFrame(rows).sort_values("f1", ascending=False)


,model,accuracy,precision,recall,f1
2,roberta-base QQP,0.9153,0.8732,0.9008,0.8868
1,bert-base-uncased QQP,0.9091,0.8665,0.8902,0.8782
0,bert-base-cased-finetuned-qqp,0.9084,0.8685,0.8852,0.8768
3,distilbert-base QQP,0.5339,0.4378,0.9362,0.5966


### Finding Duplicates (1 point)

Finally, it is time to use your model to find duplicate questions.
Please implement a function that takes a question and finds top-5 potential duplicates in the training set. For now, it is fine if your function is slow, as long as it yields correct results.

Showcase how your function works with at least 5 examples.

**Task 3 (1 point)**
- Implement function for finding duplicates
- Test it on several examples (at least 5)
- Check suggested duplicates and make a conclusion about model correctness

In [ ]:
<A whole lot of YOUR CODE HERE>

### Bonus: Finding Duplicates Faster (0.5 point)

Try to find a way to run the function faster than just passing over all questions in a loop. For isntance, you can form a short-list of potential candidates using a cheaper method, and then run your tranformer on that short list. If you opted for this solution, please keep both the original implementation and the optimized one - and explain briefly what is the difference there.

**Bonus Task 1 (0.5 point)**
- Speed up your implementation from "Finding Duplicates" part
- Capture both old and new implementation work time
- Describe your approach

In [ ]:
<A whole lot of YOUR CODE HERE>

### Bonus: Finding Duplicates in Old-Fashioned way (1.5 points)

In this bonus task you are supposed to use pretrained embeddings (word2vec, GloVe or fasttext) for solving the duplicates problem.

**Bonus Task 2 (1.5 points)**
- Solve Finding Duplicates problem using mentioned embeddings
- Compare old-fashioned solution to previous ones (quality, speed, etc.)
- Make a small report (up to 5 steps, results and conclusions) on work done in this part

In [ ]:
<A whole lot of YOUR CODE HERE>